# Dynamic Mode Decomposition on Synthetic Pendulum Video

This notebook applies Dynamic Mode Decomposition to a synthetic pendulum video sequence.

The experiment uses a controlled visual system where the true motion is known. The pendulum is useful because its motion is smooth, periodic, and low-dimensional, while the video frames are high-dimensional pixel observations.

The main DMD state representation in this notebook will be a flattened grayscale frame. The synthetic generator creates RGB images for visualization, but DMD is applied to grayscale frames so the state vector has dimension $h \cdot w$ instead of $3hw$. This keeps the experiment simpler and focuses the model on motion structure rather than color-channel variation.

The notebook proceeds through data generation, ground-truth inspection, fit/forecast window selection, snapshot construction, singular-value analysis, DMD fitting, eigenvalue and frequency interpretation, mode visualization, reconstruction, held-out forecasting, background/foreground separation, bob-centroid analysis, coordinate-level forecasting, and rank sensitivity.

The truncation rank $r$ is not fixed at the start. It will be chosen after inspecting the singular values, and later the notebook will compare several ranks to show how $r$ affects reconstruction, forecasting, mode quality, and frequency estimates.


In [ ]:
# Imports and Variables
import sys
from pathlib import Path
import json
from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name != "DMD":
    raise RuntimeError(f"Expected to run notebook from the DMD folder, got: {PROJECT_DIR}")

HELPERS_DIR = PROJECT_DIR / "helpers"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "synthetic_pendulum"

# Set to True to regenerate the synthetic dataset from scratch.
OVERWRITE_DATA = True

if str(HELPERS_DIR) not in sys.path:
    sys.path.append(str(HELPERS_DIR))

DOCS_IMAGE_DIR = PROJECT_DIR / "docs" / "images"
DOCS_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# Verify proper environment is active
print(sys.executable)
print(Path.cwd())

## 1. Generate a Synthetic Pendulum Sequence

A helper script generates RGB video frames of a pendulum swinging from a fixed pivot.

Each frame contains:

- a fixed background,
- a pendulum arm,
- and a circular bob.

The generator also saves ground-truth information for every frame, including:

- frame index,
- time,
- pendulum angle,
- angular velocity,
- pivot coordinates,
- bob-center coordinates,
- frame path,
- foreground mask path,
- and bob-only mask path.

This gives us both:

- high-dimensional image data for DMD,
- and known ground truth for evaluating motion and forecasting.

The pendulum bob is the circular weight at the end of the arm. The pivot is the fixed point from which the arm rotates.

The coordinate convention is the standard image convention:

- $(0, 0)$ is the top-left corner of the image,
- $x$ increases to the right,
- $y$ increases downward.

This matters when plotting the bob path, because image coordinates are vertically flipped relative to the usual Cartesian coordinate system.


In [ ]:
from make_synthetic_pendulum import PendulumConfig, generate_sequence

config = PendulumConfig(
    num_frames=160,
    fps=30.0,
    frame_width=256,
    frame_height=256,
    amplitude_degrees=26.0,
    period_seconds=2.0,
    damping_per_second=0.0,
    background_mode="plain",
    noise_std=0.0,
    random_seed=42,
)

summary = generate_sequence(
    config=config,
    output_dir=OUTPUT_DIR,
    overwrite=OVERWRITE_DATA,
    save_gif=True,
    readme_gif_path=OUTPUT_DIR / "readme_friendly" / "synthetic_pendulum_preview.gif",
    save_readme_gif=True,
)

summary

## 2. Inspect the Data Before Modeling

Before fitting DMD, the synthetic dataset should be inspected visually and numerically.

Useful initial plots include:

- sample video frames,
- pendulum angle versus time,
- bob $x$ and $y$ coordinates versus time,
- bob path in image coordinates,
- and the fit/forecast split.

This step is important because it establishes what the model is being asked to learn.

The video is high-dimensional because each frame contains many pixels. But the underlying motion is low-dimensional because the pendulum bob is primarily governed by a periodic angle.

That is exactly the kind of situation where DMD should be a reasonable tool: high-dimensional observations with coherent low-dimensional dynamics underneath.


In [ ]:
metadata_path = OUTPUT_DIR / "metadata.json"
ground_truth_path = OUTPUT_DIR / "ground_truth.csv"

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

ground_truth_df = pd.read_csv(ground_truth_path)

display(ground_truth_df.head())
print(json.dumps(metadata["suggested_split"], indent=2))
print("-" * 50)
print(json.dumps(metadata["config"], indent=2))

In [ ]:
import visualization

frame_paths = [OUTPUT_DIR / path for path in ground_truth_df["frame_path"]]

visualization.show_sample_frames(
    frame_paths=frame_paths,
    frame_indices=[0, len(frame_paths) // 4, len(frame_paths) // 2, 3 * len(frame_paths) // 4],
    figsize=(12, 3),
    save_path=DOCS_IMAGE_DIR / "dmd_pendulum_sample_frames.png",
)

visualization.plot_ground_truth_signals(
    ground_truth_df,
    save_path=DOCS_IMAGE_DIR / "dmd_pendulum_ground_truth_signals.png",
)

visualization.plot_bob_path(
    ground_truth_df,
    figsize=(8, 4),
    show_frame_order=True,
    annotation_mode="frame",
    save_path=DOCS_IMAGE_DIR / "dmd_pendulum_bob_path.png",
)

## 3. Choose a Fitting Window, Forecast Window, and Time Step

The video will be split into a fitting window and a forecast window.

DMD sees only the fitting window. Future frames are held out so that forecasting is a genuine extrapolation test.

For example, if the video has $N$ total frames, we might use the first 70 percent for fitting and the remaining 30 percent for forecasting.

The fitting snapshots are used to build $X$ and $X'$. The held-out snapshots are used only for evaluation.

Although this notebook uses fixed fitting and forecast windows, DMD can also be recomputed on a rolling window as new snapshots arrive, which is one way to adapt the learned linear model over time.

The key choices are:

1. the number of frames used for fitting,
2. the number of frames held out for forecasting,
3. the effective time step $\Delta t$,
4. whether to use every frame or a downsampled sequence.

For the synthetic pendulum, these choices are interpretable. Ideally, the fitting window should include enough motion for DMD to identify the dominant oscillation. If the fitting window is too short, DMD may see only part of the swing and produce poor frequency estimates.

The generated metadata stores each window as inclusive frame endpoints. For example, `[0, 111]` means frames `0` through `111`.

For array slicing, the notebook converts these to Python's half-open convention. Therefore, the fitting window `[0, 111]` becomes `0:112`, and the forecast window `[112, 159]` becomes `112:160`.


In [ ]:
# Part 3: Choose fitting window, forecast window, and time step

split = metadata["suggested_split"]
config_metadata = metadata["config"]

fps = float(config_metadata["fps"])
dt = 1.0 / fps

# The metadata stores start/end frame indices inclusively.
fit_start_index, fit_last_index = split["fit_frames"]
forecast_start_index, forecast_last_index = split["forecast_frames"]

# Convert inclusive metadata endpoints into Python's half-open slicing convention.
fit_end_index = fit_last_index + 1
forecast_end_index = forecast_last_index + 1

fit_frame_count = fit_end_index - fit_start_index # this is 'm' below
forecast_frame_count = forecast_end_index - forecast_start_index
total_frame_count = len(ground_truth_df)

fit_start_time = fit_start_index * dt
fit_end_time = fit_end_index * dt
forecast_start_time = forecast_start_index * dt
forecast_end_time = forecast_end_index * dt

split_summary_df = pd.DataFrame(
    [
        {
            "window": "fit",
            "start_frame": fit_start_index,
            "last_frame_inclusive": fit_last_index,
            "slice_end_exclusive": fit_end_index,
            "frame_count": fit_frame_count,
            "start_time_seconds": fit_start_time,
            "end_time_seconds": fit_end_time,
        },
        {
            "window": "forecast",
            "start_frame": forecast_start_index,
            "last_frame_inclusive": forecast_last_index,
            "slice_end_exclusive": forecast_end_index,
            "frame_count": forecast_frame_count,
            "start_time_seconds": forecast_start_time,
            "end_time_seconds": forecast_end_time,
        },
    ]
)

display(split_summary_df)

print(f"fps: {fps}")
print(f"dt: {dt:.6f} seconds/frame")
print(f"total frames: {total_frame_count}")
print(f"fit frames: {fit_start_index} to {fit_last_index} ({fit_frame_count} frames)")
print(f"forecast frames: {forecast_start_index} to {forecast_last_index} ({forecast_frame_count} frames)")

assert fit_start_index == 0
assert fit_end_index == forecast_start_index
assert forecast_end_index <= total_frame_count
assert fit_frame_count > 1
assert forecast_frame_count > 0

In [ ]:
visualization.plot_ground_truth_signals(
    ground_truth_df,
    fit_end_index=fit_end_index,
    save_path=DOCS_IMAGE_DIR / "dmd_pendulum_ground_truth_fit_forecast_split.png",
)

## 4. Build the DMD Snapshot Matrices

DMD is fit to grayscale frames rather than RGB frames. This reduces the state dimension from $3hw$ to $hw$, which makes the decomposition cheaper and keeps the interpretation focused on brightness/motion structure. Each frame is flattened into a vector and stacked into a snapshot matrix.

If each frame has height $h$ and width $w$, then each state vector has dimension

$$
n = h \cdot w.
$$

So each frame becomes

$$
\vec{x}_k \in \mathbb{R}^{n}.
$$

The full video sequence becomes a matrix whose columns are frames:

$$
X_{\text{full}}
=
[\vec{x}_1 \; \vec{x}_2 \; \cdots \; \vec{x}_{N}],
$$

where $N$ is the total number of frames.

For fitting DMD, we choose a fitting window containing $m$ snapshots. From those snapshots, we form

$$
X = [\vec{x}_1 \; \vec{x}_2 \; \cdots \; \vec{x}_{m-1}]
$$

and

$$
X' = [\vec{x}_2 \; \vec{x}_3 \; \cdots \; \vec{x}_{m}].
$$

DMD then estimates an approximate linear time-advance relationship:

$$
X' \approx AX.
$$

In this video setting:

- $m$ is the number of snapshots in the DMD fitting window,
- $\Delta t$ is the time between consecutive frames,
- and $n$ is the number of pixels in each flattened frame.

The value of $\Delta t$ is determined by the frame rate:

$$
\Delta t = \frac{1}{\text{fps}}.
$$

If frames are skipped or downsampled, then $\Delta t$ must be adjusted accordingly.

A key modeling choice is how we define the state vector. The old saying “garbage in, garbage out” applies here: DMD can only learn dynamics from the representation we give it.

In this notebook, the main state representation is a flattened grayscale frame. This is simple and visually interpretable, but it is not the only possible choice. We could also use RGB frames, cropped frames, downsampled frames, foreground masks, bob-only masks, or even low-dimensional coordinates such as the bob center.

For the full-frame DMD experiment, each DMD prediction is another vector in $\mathbb{R}^n$. Since $n = h \cdot w$, the predicted vector can be reshaped back into an $h \times w$ image and compared directly to the true future frame.

The sampling rate also matters. In the original financial version of this project, different bar sizes acted like different temporal views of the data. In the video setting, a similar idea would be to use every frame, every second frame, or every $q$-th frame. Changing this sampling changes the effective $\Delta t$ and therefore changes the physical meaning of one DMD time step.

In this notebook, we begin with a simple single-frame-rate setup for clarity. Later extensions could compare multiple temporal resolutions or use DMD outputs as features for another model, but the main goal here is to evaluate DMD directly through reconstruction, forecasting, mode interpretation, and trajectory comparison.

In [ ]:
# Part 4: Build the DMD snapshot matrices

def load_grayscale_frame_as_float(frame_path):
    """Load one RGB frame as a grayscale float array in [0, 1]."""
    with Image.open(frame_path) as image:
        grayscale_image = image.convert("L")
        grayscale_array = np.asarray(grayscale_image, dtype=np.float64) / 255.0

    return grayscale_array


# Load all frames as grayscale images.
grayscale_frames = [load_grayscale_frame_as_float(path) for path in frame_paths]

# Check image dimensions.
h, w = grayscale_frames[0].shape
n = h * w
N = len(grayscale_frames)

# Stack flattened frames as columns.
# Shape convention:
# X_full has shape (number of pixels, number of frames) = (n, N)
X_full = np.column_stack([frame.reshape(-1) for frame in grayscale_frames])

# The fitting window contains m snapshots.
m = fit_frame_count

# Extract fitting and held-out forecast windows.
X_fit_full = X_full[:, fit_start_index:fit_end_index]
X_forecast_true = X_full[:, forecast_start_index:forecast_end_index]

# Build the paired DMD snapshot matrices.
# X contains snapshots 1 through m-1.
# X_prime contains snapshots 2 through m.
X = X_fit_full[:, :-1]
X_prime = X_fit_full[:, 1:]

snapshot_summary_df = pd.DataFrame(
    [
        {
            "quantity": "h",
            "meaning": "frame height in pixels",
            "value": h,
        },
        {
            "quantity": "w",
            "meaning": "frame width in pixels",
            "value": w,
        },
        {
            "quantity": "n = h * w",
            "meaning": "flattened grayscale state dimension",
            "value": n,
        },
        {
            "quantity": "N",
            "meaning": "total number of video frames",
            "value": N,
        },
        {
            "quantity": "m",
            "meaning": "number of snapshots in the DMD fitting window",
            "value": m,
        },
        {
            "quantity": "Delta t",
            "meaning": "seconds per frame",
            "value": dt,
        },
    ]
)

display(snapshot_summary_df)

print(f"X_full shape:       {X_full.shape} = (n, N)")
print(f"X_fit_full shape:   {X_fit_full.shape} = (n, m)")
print(f"X shape:            {X.shape} = (n, m - 1)")
print(f"X_prime shape:      {X_prime.shape} = (n, m - 1)")
print(f"X_forecast_true shape: {X_forecast_true.shape}")

assert X_full.shape == (n, N)
assert X_fit_full.shape == (n, m)
assert X.shape == (n, m - 1)
assert X_prime.shape == (n, m - 1)
assert X_forecast_true.shape[1] == forecast_frame_count

In [ ]:
# Plot some frames as a sanity check

fig, axes = plt.subplots(1, 3, figsize=(10, 3))

example_indices = [
    fit_start_index,
    fit_end_index - 1,
    forecast_start_index,
]

titles = [
    f"Fit start\nframe {fit_start_index}",
    f"Fit end\nframe {fit_end_index - 1}",
    f"Forecast start\nframe {forecast_start_index}",
]

for ax, frame_index, title in zip(axes, example_indices, titles):
    ax.imshow(grayscale_frames[frame_index], cmap="gray")
    ax.set_title(title)
    ax.axis("off")

fig.suptitle("Grayscale frames used for DMD state vectors")
fig.tight_layout()

save_path = DOCS_IMAGE_DIR / "dmd_pendulum_grayscale_snapshot_examples.png"
fig.savefig(save_path, dpi=160, bbox_inches="tight")
plt.show()

## 5. Plot Singular Values and Choose a Rank

Before fitting DMD, we examine the singular values of the snapshot matrix. The main rank is chosen here rather than fixed at the beginning, because the singular-value spectrum gives evidence about how much low-rank structure is present.

The singular values show how much energy or variation is captured by each singular direction. A sharp decay suggests low-rank structure.

For the pendulum video, much of the variation should come from a small number of patterns:

- static background,
- the arm and bob,
- left-right swing motion,
- and phase-shifted oscillatory structure.

The rank $r$ controls how many singular directions are retained.

The notebook will begin with a main rank, then later compare multiple ranks.

Example candidate ranks might include:

$$
r \in \{2, 4, 8, 16\}.
$$

The rank is not just a technical parameter. It affects both reconstruction and forecasting.

A low rank may underfit. A high rank may reconstruct observed frames better but include weak or unstable modes that harm future prediction.


In [ ]:
# Part 5: Examine singular values before choosing a DMD rank

U_full, singular_values, Vh_full = np.linalg.svd(X, full_matrices=False)

singular_energy = singular_values**2
energy_fraction = singular_energy / singular_energy.sum()
cumulative_energy_fraction = np.cumsum(energy_fraction)

rank_indices = np.arange(1, len(singular_values) + 1)

next_value_ratio = np.full_like(singular_values, fill_value=np.nan, dtype=np.float64)
next_value_ratio[:-1] = singular_values[1:] / singular_values[:-1]

rank_summary_df = pd.DataFrame(
    {
        "rank": rank_indices,
        "singular_value": singular_values,
        "relative_to_first": singular_values / singular_values[0],
        "energy_fraction": energy_fraction,
        "cumulative_energy_fraction": cumulative_energy_fraction,
        "next_value_ratio": next_value_ratio,
    }
)

display(rank_summary_df.head(50))

candidate_ranks = [1, 2, 4, 8, 16, 24, 32]
candidate_ranks = [candidate_rank for candidate_rank in candidate_ranks if candidate_rank <= len(singular_values)]

candidate_rank_summary_df = rank_summary_df.loc[
    rank_summary_df["rank"].isin(candidate_ranks),
    [
        "rank",
        "singular_value",
        "relative_to_first",
        "energy_fraction",
        "cumulative_energy_fraction",
    ],
].reset_index(drop=True)

display(candidate_rank_summary_df)

numerical_tolerance = singular_values[0] * 1e-12
estimated_numerical_rank = int(np.sum(singular_values > numerical_tolerance))

print(f"Number of singular values: {len(singular_values)}")
print(f"Estimated numerical rank using tolerance {numerical_tolerance:.3e}: {estimated_numerical_rank}")
print(f"Largest singular value: {singular_values[0]:.6f}")
print(f"Second singular value: {singular_values[1]:.6f}")
print(f"Drop from rank 1 to rank 2: {singular_values[0] / singular_values[1]:.2f}x")

assert U_full.shape == (n, len(singular_values))
assert Vh_full.shape == (len(singular_values), m - 1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].semilogy(rank_indices, singular_values, marker="o")
axes[0].set_title("Singular Values")
axes[0].set_xlabel("Rank index")
axes[0].set_ylabel("Singular value")

axes[1].semilogy(rank_indices[1:], singular_values[1:], marker="o")
axes[1].set_title("Singular Values After Removing Rank 1")
axes[1].set_xlabel("Rank index")
axes[1].set_ylabel("Singular value")

axes[2].plot(rank_indices, cumulative_energy_fraction, marker="o")
axes[2].set_title("Cumulative Energy Captured")
axes[2].set_xlabel("Rank")
axes[2].set_ylabel("Cumulative energy fraction")
axes[2].set_ylim(0.99, 1.0005)

for ax in axes:
    for candidate_rank in candidate_ranks:
        ax.axvline(candidate_rank, linestyle=":", alpha=0.4)

fig.tight_layout()

save_path = DOCS_IMAGE_DIR / "dmd_pendulum_singular_values_examination.png"
fig.savefig(save_path, dpi=160, bbox_inches="tight")
plt.show()

The singular-value spectrum has three main regions.

First, there is an extremely large first singular value. This is expected because most pixels belong to the static scene structure. In this clean synthetic video, the plain background and average pendulum appearance dominate the total energy, so rank 1 alone captures most of the pixel intensity energy.

Second, after the first singular value, the spectrum drops sharply and then decays more gradually through the next several dozen singular values. These directions represent the lower-energy but dynamically important parts of the video: the pendulum arm, the bob, edge motion, and phase-shifted oscillatory structure. This is why cumulative energy alone is not enough to choose the DMD rank. A very low rank captures most pixel energy, but it may mostly capture the static background rather than the moving pendulum dynamics.

Third, the singular values drop to near numerical zero after roughly rank 33. That means the fitting-window snapshot matrix has an effective numerical rank around that value. However, choosing the full numerical rank is not necessarily the best modeling choice. A high rank may improve reconstruction of observed frames, but it can also make the DMD model less compact and may include weak modes that are not important for the main motion.

For the main DMD experiment, we want a rank that is large enough to include the dominant motion structure but still small enough to keep the model interpretable. Rank $r = 8$ is a reasonable first choice: it keeps the large static component plus several leading motion-related directions, while avoiding the much larger near-full-rank model. Later, the rank sensitivity section will compare this choice against smaller and larger ranks.

In [ ]:
# Main DMD rank chosen after examining the singular-value spectrum.
r = 8

print(f"Chosen main DMD rank: r = {r}")
print(f"Cumulative energy captured by rank {r}: {cumulative_energy_fraction[r - 1]:.6f}")
print(f"Singular value at rank {r}: {singular_values[r - 1]:.6f}")
print(f"Next singular value after rank {r}: {singular_values[r]:.6f}")

assert 1 <= r <= len(singular_values)

## 6. Fit DMD

DMD is fit using a rank-$r$ truncated SVD of the snapshot matrix.

The main DMD quantities are:

- $U_r$, $\Sigma_r$, and $V_r$ from the truncated SVD,
- the reduced operator $\tilde{A}$,
- the reduced eigenvectors $W$,
- the DMD eigenvalues $\Lambda$,
- and the DMD modes $\Phi$.


In [ ]:
# Part 6 code

## 7. Interpret Eigenvalues and Frequencies

After fitting, the eigenvalues can be plotted in the complex plane.

Important interpretations include:

- eigenvalues near the unit circle correspond to persistent behavior,
- eigenvalues inside the unit circle correspond to decay,
- eigenvalues outside the unit circle correspond to growth,
- complex conjugate pairs correspond to oscillatory behavior,
- eigenvalue angles determine oscillation frequencies.

For the pendulum, the most interesting result is whether DMD identifies a dominant oscillatory mode whose frequency matches the known swing period.

The discrete eigenvalues can be converted to continuous-time quantities using

$$
\omega_i = \frac{\log(\lambda_i)}{\Delta t}.
$$

Then

$$
f_i = \frac{\operatorname{Im}(\omega_i)}{2\pi}
$$

gives frequency in cycles per second, and

$$
T_i = \frac{1}{|f_i|}
$$

gives the corresponding period.

A strong result would show a dominant DMD period close to the true synthetic pendulum period.

In [ ]:
# part 7 code

## 8. Visualize DMD Modes

The DMD modes are high-dimensional vectors. Since each mode has the same dimension as a flattened frame, a mode can be reshaped back into image form.

This allows us to inspect what DMD learned.

Useful mode visualizations include:

- persistent or low-frequency modes,
- real parts of dominant oscillatory modes,
- imaginary parts of dominant oscillatory modes,
- and mode magnitudes.

For video data, this is one of the most valuable parts of DMD. The model is not just producing a forecast; it is also producing spatial patterns tied to temporal behavior.

In this pendulum example, we expect some modes to correspond mostly to static background structure and other modes to correspond to swinging pendulum motion.


In [ ]:
# Part 8 code

## 9. Reconstruct Observed Motion

The first evaluation is reconstruction.

Reconstruction asks:

> Can DMD reproduce the pendulum frames used during fitting?

Using the learned modes and eigenvalues, the notebook reconstructs frames in the fitting window.

The reconstructed frames can be compared to the true frames using:

- side-by-side visual comparisons,
- absolute error images,
- mean squared error,
- mean absolute error,
- and error over time.

Reconstruction quality tells us how well the selected rank-$r$ DMD model explains the observed training-window dynamics.

Good reconstruction does not necessarily imply good forecasting. A model can fit observed frames well while still extrapolating poorly.

In [ ]:
# Part 9 code

## 10. Forecast Future Frames

After fitting DMD on an initial window of frames, the model is used to extrapolate beyond the fitting window.

This gives a direct test of DMD as a forecasting method.

Starting from a recent observed state $\vec{x}_k$, we can forecast $s$ steps ahead using

$$
\vec{x}_{k+s}
\approx
\Phi \Lambda^s \vec{b}_k,
$$

where

$$
\vec{b}_k = \Phi^\dagger \vec{x}_k.
$$

The predicted future frames can be compared against held-out true frames.

Useful diagnostics include:

- true future frame versus predicted future frame,
- absolute error images,
- forecast MSE versus horizon,
- forecast MAE versus horizon,
- and visual inspection of phase drift.

Forecasting is harder than reconstruction because small errors in eigenvalue magnitude or phase accumulate over time.

For periodic motion, the forecast should remain coherent over short horizons when the learned DMD frequency is close to the true pendulum frequency.

In [ ]:
# Part 10 code

## 11. Background/Foreground Separation

DMD can also be used for a simple form of background/foreground separation.

The idea is:

- background-like content should be persistent or slowly varying,
- moving foreground content should require oscillatory or higher-frequency modes.

A simple DMD-based decomposition is:

$$
\text{background} \approx \text{low-frequency DMD reconstruction}
$$

and

$$
\text{foreground} \approx \text{original frame} - \text{background reconstruction}.
$$

For the synthetic pendulum, the stationary background should be captured by persistent modes, while the arm and bob should appear in the foreground residual.

The generator also provides true foreground masks and bob-only masks, which can be used for qualitative comparison.

This section should be interpreted carefully. DMD does not know what an object is. It does not semantically identify the bob or arm. It separates components based on temporal behavior.

In [ ]:
# part 11 code

## 12. Bob-Centroid Motion Analysis

Because the synthetic generator saves the true bob-center coordinates, the predicted motion can also be evaluated geometrically.

The ground-truth trajectory is given by the true bob coordinates:

$$
(x_{\text{bob}}(t), y_{\text{bob}}(t)).
$$

There are two related ways to use this information.

First, the true bob coordinates can be plotted directly to show the real pendulum path.

Second, a simple image-processing step can estimate the bob location from reconstructed or forecasted frames. The estimated bob path can then be compared against the true path from the generator.

Useful diagnostics include:

- true bob path versus estimated bob path,
- Euclidean centroid error over time,
- RMSE of the predicted centroid path,
- and comparison of trajectory shape.




In [ ]:
# part 12 code

## 13. Coordinate-Level DMD and Constant-Velocity Baseline

The following is a lower-dimensional coordinate-forecasting experiment. Instead of applying DMD to full frames, we can apply DMD to the bob-coordinate time series.

A simple coordinate state is

$$
\vec{z}_k =
\begin{bmatrix}
x_k \\
y_k
\end{bmatrix}.
$$

A richer delay-coordinate state is

$$
\vec{z}_k =
\begin{bmatrix}
x_k \\
y_k \\
x_{k-1} \\
y_{k-1} \\
\vdots \\
x_{k-q} \\
y_{k-q}
\end{bmatrix}.
$$

This delay-coordinate construction gives DMD temporal memory and is related to Hankel-DMD ideas.

Coordinate-level forecasting is often cleaner than full-frame forecasting because it models the low-dimensional motion directly rather than trying to forecast every pixel.

A useful baseline comparison is a constant-velocity forecast. Constant velocity may work briefly, but it should struggle near pendulum turning points where the direction of motion changes.

In [ ]:
# part 13 code

## 14. Rank Sensitivity Study

The notebook will also study how the choice of DMD rank affects results.

Candidate ranks might include:

$$
r \in \{2, 4, 8, 16\}.
$$

For each rank, useful quantities include:

- reconstruction error,
- forecast error,
- dominant DMD frequency,
- stability of eigenvalues,
- and visual quality of predicted frames.

This helps demonstrate that DMD is not just a single push-button method. The rank controls the balance between compression, reconstruction quality, and forecast stability.

The expected behavior is:

- very low rank may underfit the pendulum motion,
- moderate rank may capture the dominant background and oscillatory dynamics,
- very high rank may fit the training frames better but include unstable or less meaningful modes.

In [ ]:
# part 14 code

## Part 15: Final Summary and Limitations

A synthetic pendulum is a strong instructional example for DMD because:

- the system has coherent periodic dynamics,
- the video frames are high-dimensional but generated from low-dimensional motion,
- the true future states are known,
- the true bob trajectory is known,
- the results can be evaluated visually and numerically,
- the dominant frequency can be compared against the known pendulum period,
- and the learned modes can be reshaped into interpretable images.

This makes the experiment better aligned with DMD's strengths: learning dominant spatiotemporal structure from sequential data.

The strongest results to look for are:

- singular-value decay showing low-rank structure,
- eigenvalues on or near the unit circle,
- a dominant DMD frequency close to the true pendulum frequency,
- DMD modes that reveal static and oscillatory structure,
- accurate reconstruction inside the fitting window,
- short-horizon frame forecasts that remain visually coherent,
- forecast error increasing with horizon due to phase or eigenvalue errors,
- and coordinate-level forecasts that capture the bob path.

DMD is useful here because the observations are high-dimensional, but the underlying dynamics are coherent, structured, and low-dimensional.

The limitations are also important:

- the setup is synthetic and controlled,
- real video would introduce lighting changes, camera motion, occlusion, segmentation difficulty, and measurement noise,
- DMD is linear in the chosen state space,
- and long-horizon forecasts can degrade when small eigenvalue or phase errors accumulate.

The purpose of this notebook is therefore not to claim that DMD solves arbitrary video forecasting. The purpose is to show, step by step, how DMD connects linear algebra, dynamical systems, SVD, eigenvalues, video data, and interpretable forecasting.

## References

Content inspired by course lecture notes and material in __[1]__.

__[1]__ Steven L. Brunton; J. Nathan Kutz (2019). "Data Driven Science and Engineering: Machine Learning, Dynamical Systems, and Control". Section 3.7.

__[2]__ M. Gavish and D. L. Donoho, "The Optimal Hard Threshold for Singular Values is $4 / \sqrt{3}$" in IEEE Transactions on Information Theory, vol. 60, no. 8, pp. 5040-5053, Aug. 2014, doi: 10.1109/TIT.2014.2323359.